# 04. Model Evaluation
Evaluates the trained models (Baseline & XGBoost) using standardized metrics and generates comparison plots.

> **Important:** Since `SalePrice` is pre-log-transformed, metrics are computed on log-scale predictions,
> then converted to dollar-scale for interpretability.

In [ ]:
import sys, os

# --- Colab / Windows path bootstrap ---
if 'google.colab' in sys.modules:
    %cd /content/house-pridiction
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from src.models.xgboost_model import XGBoostModel
from src.config import PathConfig, ModelConfig

os.makedirs(os.path.join(PathConfig.REPORTS_DIR, 'figures'), exist_ok=True)
print(f"✅ Project root: {os.getcwd()}")

## 1. Load Processed Data

In [ ]:
if not os.path.exists(PathConfig.PROCESSED_DATA):
    raise FileNotFoundError(f"❌ Processed data not found: {PathConfig.PROCESSED_DATA}")

data = pd.read_csv(PathConfig.PROCESSED_DATA)
X = data.drop(columns=[ModelConfig.TARGET_COL])

# SalePrice is already log-transformed — use directly as target
y_true_log = data[ModelConfig.TARGET_COL]
y_true_dollar = np.expm1(y_true_log)  # For dollar-scale reporting

print(f"✅ Data loaded: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target range: ${y_true_dollar.min():,.0f} – ${y_true_dollar.max():,.0f}")

## 2. Evaluate XGBoost (Production Model)

In [ ]:
results = {}

xgb_path = os.path.join(PathConfig.MODELS_DIR, 'model.pkl')
if not os.path.exists(xgb_path):
    print(f"❌ XGBoost model not found. Run notebook 03 first.")
else:
    xgb_wrapper = XGBoostModel()
    xgb_wrapper.load(xgb_path)
    y_pred_log_xgb = xgb_wrapper.predict(X)
    y_pred_dollar_xgb = np.expm1(y_pred_log_xgb)

    rmse_log  = np.sqrt(mean_squared_error(y_true_log, y_pred_log_xgb))
    mae_log   = mean_absolute_error(y_true_log, y_pred_log_xgb)
    r2        = r2_score(y_true_log, y_pred_log_xgb)
    rmse_dol  = np.sqrt(mean_squared_error(y_true_dollar, y_pred_dollar_xgb))
    mae_dol   = mean_absolute_error(y_true_dollar, y_pred_dollar_xgb)
    results['XGBoost'] = rmse_log

    print(f"📈 XGBoost Evaluation Results:")
    print(f"   R² Score        : {r2:.4f}")
    print(f"   RMSE (log)      : {rmse_log:.4f}")
    print(f"   MAE  (log)      : {mae_log:.4f}")
    print(f"   RMSE (dollars)  : ${rmse_dol:,.0f}")
    print(f"   MAE  (dollars)  : ${mae_dol:,.0f}")

## 3. Evaluate Baseline (Linear Regression)

In [ ]:
baseline_path = os.path.join(PathConfig.MODELS_DIR, 'baseline_lr.joblib')
if not os.path.exists(baseline_path):
    print("❌ Baseline model not found. Run notebook 03 first.")
else:
    baseline = joblib.load(baseline_path)
    y_pred_log_base = baseline.predict(X)
    y_pred_dollar_base = np.expm1(y_pred_log_base)

    rmse_log_base = np.sqrt(mean_squared_error(y_true_log, y_pred_log_base))
    r2_base       = r2_score(y_true_log, y_pred_log_base)
    mae_dol_base  = mean_absolute_error(y_true_dollar, y_pred_dollar_base)
    results['Baseline'] = rmse_log_base

    print(f"📈 Baseline Evaluation Results:")
    print(f"   R² Score        : {r2_base:.4f}")
    print(f"   RMSE (log)      : {rmse_log_base:.4f}")
    print(f"   MAE  (dollars)  : ${mae_dol_base:,.0f}")

## 4. Actual vs Predicted Plot (XGBoost)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Scatter: Actual vs Predicted (dollar scale) ---
axes[0].scatter(y_true_dollar, y_pred_dollar_xgb, alpha=0.3, color='steelblue', s=10)
lims = [min(y_true_dollar.min(), y_pred_dollar_xgb.min()),
        max(y_true_dollar.max(), y_pred_dollar_xgb.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('XGBoost: Actual vs Predicted (Dollar Scale)')
axes[0].legend()

# --- Residuals ---
residuals = y_true_dollar - y_pred_dollar_xgb
axes[1].hist(residuals, bins=60, color='darkorange', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('XGBoost: Residuals Distribution')

plt.tight_layout()
fig_path = os.path.join(PathConfig.REPORTS_DIR, 'figures/xgb_eval.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"✅ Plot saved to: {fig_path}")

## 5. Model Comparison: RMSE (Log Scale)

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#1a9641' if m == 'XGBoost' else '#d73027' for m in results.keys()]
    bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor='white', width=0.5)

    for bar, val in zip(bars, results.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

    ax.set_title('Model Comparison: RMSE (Log Scale) — Lower is Better')
    ax.set_ylabel('RMSE (Log Scale)')
    ax.set_ylim(0, max(results.values()) * 1.2)

    plt.tight_layout()
    cmp_path = os.path.join(PathConfig.REPORTS_DIR, 'figures/model_comparison.png')
    plt.savefig(cmp_path, dpi=150)
    plt.show()
    print(f"✅ Comparison plot saved to: {cmp_path}")

## 6. Multi-Sample Inference Verification

In [ ]:
print(f"{'Sample':<8} | {'Predicted ($)':<15} | {'Actual ($)':<15} | {'Error (%)':<10}")
print("-" * 60)

samples = data.sample(5, random_state=42)
for i, (idx, row) in enumerate(samples.iterrows()):
    X_sample = pd.DataFrame([row.drop(ModelConfig.TARGET_COL)])
    y_actual_dollar = np.expm1(row[ModelConfig.TARGET_COL])
    y_pred_dollar   = np.expm1(xgb_wrapper.predict(X_sample)[0])
    error_pct = abs(y_pred_dollar - y_actual_dollar) / y_actual_dollar * 100
    print(f"#{i+1:<7} | ${y_pred_dollar:13,.2f} | ${y_actual_dollar:13,.2f} | {error_pct:8.2f}%")

print("\n✅ Evaluation complete.")